# SQLite Avanzado
### Fundamentos de Gestión de Datos · TECSUP · Semana 3 · Laboratorio Dirigido D3

**Docente:** Pilar Rocío Sayán Mejía · **Motor:** SQLite · **Entorno:** Google Colab (Python)

| | |
|---|---|
| **Nivel** | Avanzado (3 de 3: Básico → Intermedio → Avanzado) |
| **Requisito** | Haber terminado los cuadernos *SQLite Básico* y *SQLite Intermedio* |
| **Base de datos** | Lab D3: `clientes`, `productos` y `pedidos` (ventas de una empresa de Cusco) |
| **Sentencias que aprenderás** | `RIGHT JOIN`, `FULL JOIN`, subconsultas, `CASE`, `UNION`, vistas (`CREATE VIEW`), buenas prácticas |
| **Tiempo estimado** | 100 minutos |

## Cómo trabajar este cuaderno

1. Ejecuta las celdas **en orden**, de arriba hacia abajo.
2. No necesitas escribir código nuevo: **lee la consulta, ejecútala e interpreta** el resultado.
3. El cuaderno puede ejecutarse completo varias veces sin duplicar datos.
4. Todos los resultados que se comentan en el texto se obtienen al ejecutar las celdas. Si tu resultado es distinto, revisa que ejecutaste todo en orden.

## 1. Base de datos del Lab D3: Clientes, Pedidos y Productos

Antes de escribir un `JOIN` hay que entender el **modelo de datos**. El Lab D3 tiene tres tablas relacionadas que simulan el sistema de ventas de una empresa en Cusco.

```
clientes (id_cliente)  ◄──  pedidos (id_cliente, id_producto)  ──►  productos (id_producto)

Un cliente  → MUCHOS pedidos.        Un producto → MUCHOS pedidos.
La tabla pedidos conecta clientes con productos.
Monto de cada pedido = pedidos.cantidad × productos.precio
```

### Preparación del entorno

Ejecuta la celda de preparación: carga `sqlite3` y `pandas` y abre la conexión `conn`. Las sentencias que cambian la base (`CREATE`, `INSERT`...) se escriben con `conn.executescript(...)` y las consultas `SELECT` con `pd.read_sql_query(..., conn)`.

In [1]:
# CELDA DE PREPARACIÓN: ejecútala una sola vez al inicio.
import sqlite3
import pandas as pd

conn = sqlite3.connect("labd3.db")      # crea (o abre) el archivo de la base de datos
conn.execute("PRAGMA foreign_keys = ON")   # SQLite necesita esta línea para respetar las claves foráneas

print("Versión de SQLite:", sqlite3.sqlite_version)

Versión de SQLite: 3.45.1


### Crear las tres tablas

Primero las tablas **padre** (`clientes`, `productos`) y después la **hija** (`pedidos`). `DROP TABLE IF EXISTS` permite repetir la celda sin errores.

In [2]:
conn.executescript("""
DROP VIEW  IF EXISTS v_productos_ranking;
DROP VIEW  IF EXISTS v_ventas_cliente;
DROP TABLE IF EXISTS pedidos;
DROP TABLE IF EXISTS productos;
DROP TABLE IF EXISTS clientes;

CREATE TABLE clientes (
    id_cliente  INTEGER PRIMARY KEY AUTOINCREMENT,   -- PK: identificador único
    nombre      TEXT NOT NULL,                       -- nombre completo
    ciudad      TEXT DEFAULT 'Cusco',                -- ciudad del cliente
    email       TEXT UNIQUE                          -- correo único (puede estar vacío)
);

CREATE TABLE productos (
    id_producto INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre      TEXT NOT NULL,
    categoria   TEXT NOT NULL,                       -- Software, Hardware o Servicio
    precio      REAL NOT NULL                        -- precio unitario en soles
);

CREATE TABLE pedidos (
    id_pedido   INTEGER PRIMARY KEY AUTOINCREMENT,
    fecha       TEXT NOT NULL,                       -- formato 'AAAA-MM-DD'
    cantidad    INTEGER NOT NULL,                    -- unidades compradas
    id_cliente  INTEGER NOT NULL,                    -- FK → clientes
    id_producto INTEGER NOT NULL,                    -- FK → productos
    FOREIGN KEY (id_cliente)  REFERENCES clientes(id_cliente),
    FOREIGN KEY (id_producto) REFERENCES productos(id_producto)
);
""")
print("Tablas creadas correctamente")

Tablas creadas correctamente


### Insertar los datos de ejemplo

10 clientes, 6 productos y 15 pedidos. **Marco Salas** no tiene correo ni ha hecho ningún pedido: nos servirá para entender los `JOIN`.

In [3]:
conn.executescript("""
INSERT INTO clientes (nombre, ciudad, email) VALUES
    ('María Quispe',    'Cusco',    'mquispe@mail.com'),   -- id = 1
    ('Carlos Ttito',    'Cusco',    'cttito@mail.com'),    -- id = 2
    ('Ana Mamani',      'Puno',     'amamani@mail.com'),   -- id = 3
    ('Pedro Huanca',    'Cusco',    'phuanca@mail.com'),   -- id = 4
    ('Lucía Condori',   'Arequipa', 'lcondori@mail.com'),  -- id = 5
    ('José Apaza',      'Cusco',    'japaza@mail.com'),    -- id = 6
    ('Sofía Ccopa',     'Lima',     'sccopa@mail.com'),    -- id = 7
    ('Roberto Huallpa', 'Cusco',    'rhuallpa@mail.com'),  -- id = 8
    ('Diana Flores',    'Puno',     'dflores@mail.com'),   -- id = 9
    ('Marco Salas',     'Cusco',    NULL);                 -- id = 10, sin correo

INSERT INTO productos (nombre, categoria, precio) VALUES
    ('Licencia Office 365', 'Software', 450.00),    -- id = 1
    ('Laptop Dell i7',      'Hardware', 3200.00),   -- id = 2
    ('Soporte Técnico',     'Servicio', 250.00),    -- id = 3
    ('Antivirus Anual',     'Software', 180.00),    -- id = 4
    ('Disco Externo 2TB',   'Hardware', 320.00),    -- id = 5
    ('Capacitación SQLite', 'Servicio', 500.00);    -- id = 6

INSERT INTO pedidos (fecha, cantidad, id_cliente, id_producto) VALUES
    ('2024-01-10', 2, 1, 1),   -- María compra 2 licencias Office
    ('2024-01-15', 1, 2, 2),   -- Carlos compra 1 laptop
    ('2024-01-20', 3, 1, 4),   -- María compra 3 antivirus
    ('2024-02-05', 1, 3, 3),   -- Ana pide soporte técnico
    ('2024-02-10', 2, 4, 1),   -- Pedro compra 2 licencias
    ('2024-02-18', 1, 5, 6),   -- Lucía: capacitación
    ('2024-03-01', 1, 2, 5),   -- Carlos: disco externo
    ('2024-03-10', 4, 6, 4),   -- José: 4 antivirus
    ('2024-03-22', 1, 1, 2),   -- María: laptop
    ('2024-04-05', 2, 7, 3),   -- Sofía: 2 soportes
    ('2024-04-15', 1, 8, 1),   -- Roberto: licencia
    ('2024-05-01', 3, 4, 4),   -- Pedro: 3 antivirus
    ('2024-05-10', 1, 9, 6),   -- Diana: capacitación
    ('2024-05-20', 2, 1, 3),   -- María: 2 soportes
    ('2024-06-01', 1, 2, 6);   -- Carlos: capacitación
""")
print("Base de datos lista")

Base de datos lista


In [4]:
pd.read_sql_query("""
SELECT 'clientes' AS Tabla, COUNT(*) AS Filas FROM clientes
UNION ALL SELECT 'productos', COUNT(*) FROM productos
UNION ALL SELECT 'pedidos',   COUNT(*) FROM pedidos
""", conn)

,Tabla,Filas
0,clientes,10
1,productos,6
2,pedidos,15


## 2. Repaso: `INNER JOIN` y `LEFT JOIN`

Un `JOIN` combina filas de dos o más tablas usando una columna común. La diferencia entre tipos está en **qué filas se incluyen cuando no hay coincidencia**.

| Tipo | ¿Qué devuelve? | Filas sin pareja |
|---|---|---|
| `INNER JOIN` | Solo filas con coincidencia en **ambas** tablas | Se descartan |
| `LEFT JOIN` | **Todas** las de la izquierda + coincidencias de la derecha | `NULL` en columnas de la derecha |
| `RIGHT JOIN` | **Todas** las de la derecha + coincidencias de la izquierda | `NULL` en columnas de la izquierda |
| `FULL JOIN` | **Todas** las filas de ambas tablas | `NULL` en ambos lados |

### 2.1 `INNER JOIN`: solo lo que coincide en ambas tablas

Si un cliente no tiene pedidos, **no aparece**. Une tres tablas para ver cada pedido con su cliente, su producto y su monto.

In [5]:
pd.read_sql_query("""
SELECT p.id_pedido             AS Pedido,
       p.fecha                 AS Fecha,
       c.nombre                AS Cliente,        -- c  = alias de clientes
       pr.nombre               AS Producto,       -- pr = alias de productos
       p.cantidad              AS Qty,
       pr.precio               AS Precio_Unit,
       p.cantidad * pr.precio  AS Monto_Total     -- el monto se calcula aquí mismo
FROM   pedidos p                                  -- tabla principal
INNER JOIN clientes  c  ON p.id_cliente  = c.id_cliente
INNER JOIN productos pr ON p.id_producto = pr.id_producto
ORDER BY p.fecha, p.id_pedido
""", conn)

,Pedido,Fecha,Cliente,Producto,Qty,Precio_Unit,Monto_Total
0,1,2024-01-10,María Quispe,Licencia Office 365,2,450.0,900.0
1,2,2024-01-15,Carlos Ttito,Laptop Dell i7,1,3200.0,3200.0
2,3,2024-01-20,María Quispe,Antivirus Anual,3,180.0,540.0
3,4,2024-02-05,Ana Mamani,Soporte Técnico,1,250.0,250.0
4,5,2024-02-10,Pedro Huanca,Licencia Office 365,2,450.0,900.0
5,6,2024-02-18,Lucía Condori,Capacitación SQLite,1,500.0,500.0
6,7,2024-03-01,Carlos Ttito,Disco Externo 2TB,1,320.0,320.0
7,8,2024-03-10,José Apaza,Antivirus Anual,4,180.0,720.0
8,9,2024-03-22,María Quispe,Laptop Dell i7,1,3200.0,3200.0
9,10,2024-04-05,Sofía Ccopa,Soporte Técnico,2,250.0,500.0


### 2.2 `LEFT JOIN`: todos los de la izquierda, aunque no tengan pedidos

Devuelve **todos los clientes**, incluso los que nunca compraron. Donde no hay pedido, las columnas de pedidos quedan vacías y `COALESCE(x, 0)` convierte ese vacío en `0`.

In [6]:
pd.read_sql_query("""
SELECT c.nombre                                   AS Cliente,
       c.ciudad                                   AS Ciudad,
       COUNT(p.id_pedido)                         AS Total_Pedidos,   -- COUNT ignora los NULL
       COALESCE(SUM(p.cantidad * pr.precio), 0)   AS Total_Comprado   -- COALESCE: NULL → 0
FROM   clientes c                                 -- tabla IZQUIERDA: todos los clientes
LEFT JOIN pedidos   p  ON c.id_cliente  = p.id_cliente
LEFT JOIN productos pr ON p.id_producto = pr.id_producto
GROUP BY c.id_cliente, c.nombre, c.ciudad
ORDER BY Total_Comprado DESC, Cliente
""", conn)

,Cliente,Ciudad,Total_Pedidos,Total_Comprado
0,María Quispe,Cusco,4,5140.0
1,Carlos Ttito,Cusco,3,4020.0
2,Pedro Huanca,Cusco,2,1440.0
3,José Apaza,Cusco,1,720.0
4,Diana Flores,Puno,1,500.0
5,Lucía Condori,Arequipa,1,500.0
6,Sofía Ccopa,Lima,1,500.0
7,Roberto Huallpa,Cusco,1,450.0
8,Ana Mamani,Puno,1,250.0
9,Marco Salas,Cusco,0,0.0


> **Marco Salas aparece con 0 pedidos y S/ 0.** Con `INNER JOIN` no aparecería. Con `LEFT JOIN` sí. Esto es clave para reportes de **clientes inactivos**.

## 3. `RIGHT JOIN` y `FULL JOIN`

`RIGHT JOIN` y `FULL JOIN` se agregaron a SQLite en la **versión 3.39.0 (junio de 2022)**. Google Colab puede traer una versión anterior o posterior, así que primero comprobamos cuál tienes (si es **3.39.0 o mayor**, puedes usar también las formas nativas que se muestran en cada sección; si no, usa siempre las formas equivalentes):

In [7]:
sqlite3.sqlite_version

'3.45.1'

### 3.1 `RIGHT JOIN` = un `LEFT JOIN` con las tablas intercambiadas

Estas dos consultas devuelven **lo mismo**: todos los clientes (la tabla de la derecha en la primera consulta) con sus pedidos. La forma equivalente funciona en **cualquier** versión.

**Forma equivalente (siempre funciona):** se escribe `clientes` primero y se usa `LEFT JOIN`.

In [8]:
pd.read_sql_query("""
SELECT c.nombre AS Cliente, p.id_pedido AS Pedido
FROM   clientes c
LEFT JOIN pedidos p ON c.id_cliente = p.id_cliente
ORDER BY c.nombre, p.id_pedido
""", conn)

,Cliente,Pedido
0,Ana Mamani,4.0
1,Carlos Ttito,2.0
2,Carlos Ttito,7.0
3,Carlos Ttito,15.0
4,Diana Flores,13.0
5,José Apaza,8.0
6,Lucía Condori,6.0
7,Marco Salas,NaN
8,María Quispe,1.0
9,María Quispe,3.0


**Forma nativa** (solo si tu versión es 3.39 o mayor): `pedidos RIGHT JOIN clientes` conserva todo lo de la tabla de la derecha (`clientes`) y devuelve **exactamente lo mismo** que la consulta anterior.

```sql
SELECT c.nombre AS Cliente, p.id_pedido AS Pedido
FROM   pedidos p
RIGHT JOIN clientes c ON c.id_cliente = p.id_cliente
ORDER BY c.nombre, p.id_pedido;
```

### 3.2 `FULL JOIN`: todo de ambas tablas

Con las tablas de ventas no se nota la diferencia (todo pedido tiene un cliente). Se entiende mejor **comparando dos listas**: quiénes asistieron a un **taller** y quiénes se inscribieron a un **curso**.

In [9]:
conn.executescript("""
DROP TABLE IF EXISTS taller;
DROP TABLE IF EXISTS curso;
CREATE TABLE taller (nombre TEXT);
CREATE TABLE curso  (nombre TEXT);
INSERT INTO taller VALUES ('Ana'), ('Luis'), ('Marta');
INSERT INTO curso  VALUES ('Luis'), ('Marta'), ('Pedro');
""")
pd.read_sql_query("SELECT 'taller' AS Lista, nombre FROM taller UNION ALL SELECT 'curso', nombre FROM curso", conn)

,Lista,nombre
0,taller,Ana
1,taller,Luis
2,taller,Marta
3,curso,Luis
4,curso,Marta
5,curso,Pedro


**Forma equivalente (siempre funciona):** `FULL JOIN` = `LEFT JOIN` **+** el `LEFT JOIN` inverso, unidos con `UNION` (que elimina los duplicados de las filas que coinciden en ambos lados).

In [10]:
pd.read_sql_query("""
SELECT *
FROM (
    SELECT t.nombre AS En_Taller, k.nombre AS En_Curso
    FROM   taller t
    LEFT JOIN curso k ON t.nombre = k.nombre       -- todo lo del taller
    UNION
    SELECT t.nombre, k.nombre
    FROM   curso k
    LEFT JOIN taller t ON t.nombre = k.nombre      -- todo lo del curso
)
ORDER BY COALESCE(En_Taller, En_Curso)
""", conn)

,En_Taller,En_Curso
0,Ana,NaN
1,Luis,Luis
2,Marta,Marta
3,NaN,Pedro


**Forma nativa** (solo en 3.39 o mayor), con el mismo resultado:

```sql
SELECT t.nombre AS En_Taller, k.nombre AS En_Curso
FROM   taller t
FULL JOIN curso k ON t.nombre = k.nombre
ORDER BY COALESCE(t.nombre, k.nombre);
```

**Cómo leerlo:** *Ana* solo está en el taller (`En_Curso` vacío), *Pedro* solo en el curso (`En_Taller` vacío) y *Luis* y *Marta* están en ambos.

| Resumen de uniones en SQLite | Cómo se escribe |
|---|---|
| `INNER JOIN` | Nativo en todas las versiones |
| `LEFT JOIN` | Nativo en todas las versiones |
| `RIGHT JOIN` | Nativo desde 3.39; antes: `LEFT JOIN` con las tablas intercambiadas |
| `FULL JOIN` | Nativo desde 3.39; antes: `UNION` de dos `LEFT JOIN` |

## 4. `GROUP BY` y `HAVING`: agrupar y filtrar grupos

> **Diferencia clave:** `WHERE` filtra filas **antes** de agrupar; `HAVING` filtra grupos **después** de agrupar. Si el filtro usa una función de resumen (`SUM`, `COUNT`...), usa `HAVING`.

### 4.1 Ventas por categoría de producto

Se usa el **ticket promedio** (monto promedio por pedido), que es la cifra que tiene sentido para comparar categorías.

In [11]:
pd.read_sql_query("""
SELECT pr.categoria                          AS Categoria,
       COUNT(p.id_pedido)                    AS Num_Pedidos,
       SUM(p.cantidad * pr.precio)           AS Total_Ventas,
       ROUND(AVG(p.cantidad * pr.precio), 2) AS Ticket_Promedio
FROM   pedidos p
INNER JOIN productos pr ON p.id_producto = pr.id_producto
GROUP BY pr.categoria
ORDER BY Total_Ventas DESC
""", conn)

,Categoria,Num_Pedidos,Total_Ventas,Ticket_Promedio
0,Hardware,3,6720.0,2240.00
1,Software,6,4050.0,675.00
2,Servicio,6,2750.0,458.33


### 4.2 `HAVING`: solo categorías con ventas mayores a S/ 3 000

In [12]:
pd.read_sql_query("""
SELECT pr.categoria                AS Categoria,
       SUM(p.cantidad * pr.precio) AS Total_Ventas
FROM   pedidos p
INNER JOIN productos pr ON p.id_producto = pr.id_producto
GROUP BY pr.categoria
HAVING SUM(p.cantidad * pr.precio) > 3000     -- HAVING filtra DESPUÉS de agrupar
ORDER BY Total_Ventas DESC
""", conn)

,Categoria,Total_Ventas
0,Hardware,6720.0
1,Software,4050.0


### 4.3 `GROUP BY` con varias columnas: ventas por mes y ciudad

`SUBSTR(p.fecha, 1, 7)` toma los primeros 7 caracteres de la fecha (`'2024-05-10'` → `'2024-05'`), es decir, el año-mes.

In [13]:
pd.read_sql_query("""
SELECT SUBSTR(p.fecha, 1, 7)           AS Mes,
       c.ciudad                        AS Ciudad,
       COUNT(p.id_pedido)              AS Pedidos,
       SUM(p.cantidad * pr.precio)     AS Total
FROM   pedidos p
INNER JOIN clientes  c  ON p.id_cliente  = c.id_cliente
INNER JOIN productos pr ON p.id_producto = pr.id_producto
GROUP BY SUBSTR(p.fecha, 1, 7), c.ciudad       -- agrupa por mes Y ciudad
ORDER BY Mes, Total DESC, Ciudad
""", conn)

,Mes,Ciudad,Pedidos,Total
0,2024-01,Cusco,3,4640.0
1,2024-02,Cusco,1,900.0
2,2024-02,Arequipa,1,500.0
3,2024-02,Puno,1,250.0
4,2024-03,Cusco,3,4240.0
5,2024-04,Lima,1,500.0
6,2024-04,Cusco,1,450.0
7,2024-05,Cusco,2,1040.0
8,2024-05,Puno,1,500.0
9,2024-06,Cusco,1,500.0


## 5. Subconsultas: consultas dentro de consultas

Una **subconsulta** es un `SELECT` dentro de otro `SELECT`, entre paréntesis. Puede aparecer en el `WHERE`/`HAVING`, en el `FROM` o en el `SELECT`.

### 5.1 Subconsulta en `HAVING`: clientes que compraron más que el promedio

Primero calculamos cuánto compró cada cliente (consulta interna), luego el **promedio** de esos totales y, por último, mostramos solo a los que lo superan.

In [14]:
print("Paso 1: el promedio de compra por cliente (solo clientes con pedidos):")
pd.read_sql_query("""
SELECT ROUND(AVG(total_cliente), 2) AS Promedio_Por_Cliente
FROM (
    SELECT SUM(p2.cantidad * pr2.precio) AS total_cliente
    FROM   pedidos p2
    INNER JOIN productos pr2 ON p2.id_producto = pr2.id_producto
    GROUP BY p2.id_cliente
)
""", conn)

Paso 1: el promedio de compra por cliente (solo clientes con pedidos):


,Promedio_Por_Cliente
0,1502.22


In [15]:
print("Paso 2: los clientes que superan ese promedio:")
pd.read_sql_query("""
SELECT c.nombre                    AS Cliente,
       SUM(p.cantidad * pr.precio) AS Total_Comprado
FROM   pedidos p
INNER JOIN clientes  c  ON p.id_cliente  = c.id_cliente
INNER JOIN productos pr ON p.id_producto = pr.id_producto
GROUP BY c.id_cliente
HAVING SUM(p.cantidad * pr.precio) > (        -- se compara con una subconsulta
    SELECT AVG(total_cliente) FROM (          -- subconsulta anidada
        SELECT SUM(p2.cantidad * pr2.precio) AS total_cliente
        FROM   pedidos p2
        INNER JOIN productos pr2 ON p2.id_producto = pr2.id_producto
        GROUP BY p2.id_cliente
    )
)
ORDER BY Total_Comprado DESC
""", conn)

Paso 2: los clientes que superan ese promedio:


,Cliente,Total_Comprado
0,María Quispe,5140.0
1,Carlos Ttito,4020.0


### 5.2 Subconsulta con `IN` y `NOT IN`

`IN (subconsulta)` mantiene las filas cuyo valor aparece en el resultado de la subconsulta. `NOT IN` hace lo contrario.

In [16]:
print("Clientes que compraron algún producto de la categoría Hardware:")
pd.read_sql_query("""
SELECT nombre AS Cliente, ciudad AS Ciudad
FROM   clientes
WHERE  id_cliente IN (
    SELECT p.id_cliente
    FROM   pedidos p
    INNER JOIN productos pr ON p.id_producto = pr.id_producto
    WHERE  pr.categoria = 'Hardware'
)
ORDER BY nombre
""", conn)

Clientes que compraron algún producto de la categoría Hardware:


,Cliente,Ciudad
0,Carlos Ttito,Cusco
1,María Quispe,Cusco


In [17]:
print("Clientes que nunca han comprado:")
pd.read_sql_query("""
SELECT nombre AS Cliente, ciudad AS Ciudad
FROM   clientes
WHERE  id_cliente NOT IN (SELECT id_cliente FROM pedidos)
""", conn)

Clientes que nunca han comprado:


,Cliente,Ciudad
0,Marco Salas,Cusco


### 5.3 Subconsulta en `FROM` con `CASE`: clasificar clientes

Una subconsulta en el `FROM` actúa como una **tabla temporal**: debe llevar un **alias** (`AS resumen`). `CASE` es como un «si... entonces... si no...» dentro del `SELECT`: evalúa las condiciones en orden y devuelve el primer valor que se cumple.

In [18]:
pd.read_sql_query("""
SELECT resumen.Cliente,
       resumen.Total_Comprado,
       CASE                                          -- CASE funciona como un si / si no
           WHEN resumen.Total_Comprado >= 5000 THEN 'VIP'
           WHEN resumen.Total_Comprado >= 2000 THEN 'Frecuente'
           ELSE 'Ocasional'
       END AS Categoria_Cliente
FROM (                                               -- inicio de la subconsulta en FROM
    SELECT c.nombre                    AS Cliente,
           SUM(p.cantidad * pr.precio) AS Total_Comprado
    FROM   pedidos p
    INNER JOIN clientes  c  ON p.id_cliente  = c.id_cliente
    INNER JOIN productos pr ON p.id_producto = pr.id_producto
    GROUP BY c.id_cliente
) AS resumen                                         -- el alias es obligatorio
ORDER BY Total_Comprado DESC, Cliente
""", conn)

,Cliente,Total_Comprado,Categoria_Cliente
0,María Quispe,5140.0,VIP
1,Carlos Ttito,4020.0,Frecuente
2,Pedro Huanca,1440.0,Ocasional
3,José Apaza,720.0,Ocasional
4,Diana Flores,500.0,Ocasional
5,Lucía Condori,500.0,Ocasional
6,Sofía Ccopa,500.0,Ocasional
7,Roberto Huallpa,450.0,Ocasional
8,Ana Mamani,250.0,Ocasional


### 5.4 Subconsulta en `SELECT`: producto favorito de cada cliente

La subconsulta devuelve **un solo valor por cada cliente** (el producto del que más unidades compró). Se conecta con la consulta externa mediante `p2.id_cliente = c.id_cliente`. Si dos productos empatan en unidades, se desempata por **orden alfabético** para que el resultado siempre sea el mismo.

In [19]:
pd.read_sql_query("""
SELECT c.nombre AS Cliente,
       (SELECT pr.nombre
        FROM   pedidos p2
        INNER JOIN productos pr ON p2.id_producto = pr.id_producto
        WHERE  p2.id_cliente = c.id_cliente          -- conecta con el cliente de afuera
        GROUP BY pr.id_producto
        ORDER BY SUM(p2.cantidad) DESC, pr.nombre    -- el más comprado; empate: alfabético
        LIMIT 1) AS Producto_Favorito
FROM   clientes c
WHERE  c.id_cliente IN (SELECT DISTINCT id_cliente FROM pedidos)   -- solo clientes con pedidos
ORDER BY c.nombre
""", conn)

,Cliente,Producto_Favorito
0,Ana Mamani,Soporte Técnico
1,Carlos Ttito,Capacitación SQLite
2,Diana Flores,Capacitación SQLite
3,José Apaza,Antivirus Anual
4,Lucía Condori,Capacitación SQLite
5,María Quispe,Antivirus Anual
6,Pedro Huanca,Antivirus Anual
7,Roberto Huallpa,Licencia Office 365
8,Sofía Ccopa,Soporte Técnico


## 6. `UNION` y `UNION ALL`: juntar resultados

`UNION` une los resultados de **dos `SELECT`** en una sola lista (deben tener las mismas columnas). `UNION` **elimina las filas repetidas**; `UNION ALL` las conserva y es más rápido.

Ejemplo: una sola lista con los **clientes que compraron más de S/ 4 000** y los **clientes que nunca compraron**, con una columna que indica el grupo.

In [20]:
pd.read_sql_query("""
SELECT c.nombre AS Cliente, 'Compró más de S/ 4 000' AS Grupo
FROM   clientes c
INNER JOIN pedidos   p  ON c.id_cliente  = p.id_cliente
INNER JOIN productos pr ON p.id_producto = pr.id_producto
GROUP BY c.id_cliente
HAVING SUM(p.cantidad * pr.precio) > 4000

UNION ALL

SELECT nombre, 'Nunca compró'
FROM   clientes
WHERE  id_cliente NOT IN (SELECT id_cliente FROM pedidos)
""", conn)

,Cliente,Grupo
0,María Quispe,Compró más de S/ 4 000
1,Carlos Ttito,Compró más de S/ 4 000
2,Marco Salas,Nunca compró


## 7. Vistas (`VIEW`): guardar y reutilizar consultas

> **¿Qué es una vista?** Una consulta `SELECT` **guardada con un nombre** en la base de datos. No almacena datos: cada vez que la consultas, ejecuta el `SELECT` original. Se usa igual que una tabla: `SELECT * FROM nombre_vista`. Simplifica consultas complejas, mejora la legibilidad y se reutiliza.

### 7.1 Crear una vista de resumen de ventas por cliente

In [21]:
conn.executescript("""
CREATE VIEW v_ventas_cliente AS
SELECT c.id_cliente,
       c.nombre                    AS Cliente,
       c.ciudad                    AS Ciudad,
       COUNT(p.id_pedido)          AS Total_Pedidos,
       SUM(p.cantidad * pr.precio) AS Total_Comprado,
       MAX(p.fecha)                AS Ultima_Compra
FROM   clientes c
LEFT JOIN pedidos   p  ON c.id_cliente  = p.id_cliente      -- LEFT: incluye clientes sin pedidos
LEFT JOIN productos pr ON p.id_producto = pr.id_producto
GROUP BY c.id_cliente, c.nombre, c.ciudad
""")
print("Vista v_ventas_cliente creada")

Vista v_ventas_cliente creada


### 7.2 Usar la vista como si fuera una tabla

In [22]:
print("Toda la vista, ordenada:")
pd.read_sql_query("SELECT * FROM v_ventas_cliente ORDER BY Total_Comprado DESC, Cliente", conn)

Toda la vista, ordenada:


,id_cliente,Cliente,Ciudad,Total_Pedidos,Total_Comprado,Ultima_Compra
0,1,María Quispe,Cusco,4,5140.0,2024-05-20
1,2,Carlos Ttito,Cusco,3,4020.0,2024-06-01
2,4,Pedro Huanca,Cusco,2,1440.0,2024-05-01
3,6,José Apaza,Cusco,1,720.0,2024-03-10
4,9,Diana Flores,Puno,1,500.0,2024-05-10
5,5,Lucía Condori,Arequipa,1,500.0,2024-02-18
6,7,Sofía Ccopa,Lima,1,500.0,2024-04-05
7,8,Roberto Huallpa,Cusco,1,450.0,2024-04-15
8,3,Ana Mamani,Puno,1,250.0,2024-02-05
9,10,Marco Salas,Cusco,0,NaN,NaN


In [23]:
print("Filtrando sobre la vista: clientes de Cusco con pedidos (los 5 mayores):")
pd.read_sql_query("""
SELECT Cliente, Ciudad, Total_Comprado
FROM   v_ventas_cliente
WHERE  Ciudad = 'Cusco'
AND    Total_Pedidos > 0
ORDER BY Total_Comprado DESC
LIMIT 5
""", conn)

Filtrando sobre la vista: clientes de Cusco con pedidos (los 5 mayores):


,Cliente,Ciudad,Total_Comprado
0,María Quispe,Cusco,5140.0
1,Carlos Ttito,Cusco,4020.0
2,Pedro Huanca,Cusco,1440.0
3,José Apaza,Cusco,720.0
4,Roberto Huallpa,Cusco,450.0


### 7.3 Vista de ranking de productos

El orden (`ORDER BY`) se aplica al **consultar** la vista, no dentro de ella: dentro de una vista no se garantiza el orden.

In [24]:
conn.executescript("""
CREATE VIEW v_productos_ranking AS
SELECT pr.id_producto,
       pr.nombre                   AS Producto,
       pr.categoria                AS Categoria,
       pr.precio                   AS Precio_Unit,
       COUNT(p.id_pedido)          AS Veces_Vendido,
       SUM(p.cantidad)             AS Unidades_Total,
       SUM(p.cantidad * pr.precio) AS Ingresos_Total
FROM   productos pr
LEFT JOIN pedidos p ON pr.id_producto = p.id_producto
GROUP BY pr.id_producto
""")
pd.read_sql_query("SELECT * FROM v_productos_ranking ORDER BY Ingresos_Total DESC", conn)

,id_producto,Producto,Categoria,Precio_Unit,Veces_Vendido,Unidades_Total,Ingresos_Total
0,2,Laptop Dell i7,Hardware,3200.0,2,2,6400.0
1,1,Licencia Office 365,Software,450.0,3,5,2250.0
2,4,Antivirus Anual,Software,180.0,3,10,1800.0
3,6,Capacitación SQLite,Servicio,500.0,3,3,1500.0
4,3,Soporte Técnico,Servicio,250.0,3,5,1250.0
5,5,Disco Externo 2TB,Hardware,320.0,1,1,320.0


## 8. Buenas prácticas: alias, legibilidad y comentarios

Un buen código no es solo el que funciona: es el que **otra persona puede leer y entender sin preguntar**.

### 8.1 Alias: nombres cortos y descriptivos

Las dos consultas siguientes devuelven **exactamente las mismas 15 filas**. La segunda es mucho más fácil de leer.

In [25]:
print("MAL: sin alias, difícil de leer")
pd.read_sql_query("""
SELECT clientes.nombre, pedidos.fecha, productos.precio
FROM clientes INNER JOIN pedidos
  ON clientes.id_cliente = pedidos.id_cliente
INNER JOIN productos ON pedidos.id_producto = productos.id_producto
""", conn)

MAL: sin alias, difícil de leer


,nombre,fecha,precio
0,María Quispe,2024-01-10,450.0
1,Carlos Ttito,2024-01-15,3200.0
2,María Quispe,2024-01-20,180.0
3,Ana Mamani,2024-02-05,250.0
4,Pedro Huanca,2024-02-10,450.0
5,Lucía Condori,2024-02-18,500.0
6,Carlos Ttito,2024-03-01,320.0
7,José Apaza,2024-03-10,180.0
8,María Quispe,2024-03-22,3200.0
9,Sofía Ccopa,2024-04-05,250.0


In [26]:
print("BIEN: con alias claros y columnas renombradas")
pd.read_sql_query("""
SELECT c.nombre  AS Cliente,        -- AS renombra la columna
       p.fecha   AS Fecha_Compra,
       pr.precio AS Precio_Soles
FROM   clientes  c                  -- c  = alias corto de clientes
INNER JOIN pedidos   p  ON c.id_cliente  = p.id_cliente
INNER JOIN productos pr ON p.id_producto = pr.id_producto
""", conn)

BIEN: con alias claros y columnas renombradas


,Cliente,Fecha_Compra,Precio_Soles
0,María Quispe,2024-01-10,450.0
1,Carlos Ttito,2024-01-15,3200.0
2,María Quispe,2024-01-20,180.0
3,Ana Mamani,2024-02-05,250.0
4,Pedro Huanca,2024-02-10,450.0
5,Lucía Condori,2024-02-18,500.0
6,Carlos Ttito,2024-03-01,320.0
7,José Apaza,2024-03-10,180.0
8,María Quispe,2024-03-22,3200.0
9,Sofía Ccopa,2024-04-05,250.0


### 8.2 Comentarios en SQLite

```sql
-- Esto es un comentario de una línea

/* Esto es un comentario
   de varias líneas.
   Útil para describir el propósito de la consulta. */
```

> En **Python** los comentarios se escriben con `#`. Dentro del texto de una consulta (entre comillas triples) se escriben con `--`. **No los mezcles.**

### 8.3 Tabla de buenas prácticas de formato

| Práctica | Evitar | Preferir |
|---|---|---|
| Palabras clave | `select * from pedidos` | `SELECT * FROM pedidos` |
| Una cláusula por línea | `SELECT a FROM b WHERE c GROUP BY d` | cada cláusula (`SELECT`, `FROM`, `WHERE`...) en su propia línea |
| Indentación en subconsultas | `FROM(SELECT y FROM z)` | `FROM (` + subconsulta sangrada en varias líneas + `)` |
| Nombrar columnas calculadas | `SUM(cantidad * precio)` | `SUM(cantidad * precio) AS Total` |
| Alias de tablas | `INNER JOIN clientes ON clientes.id = ...` | `INNER JOIN clientes c ON c.id = ...` |
| Desempate en `ORDER BY` | `ORDER BY Monto DESC LIMIT 5` | `ORDER BY Monto DESC, Cliente LIMIT 5` (así el resultado es siempre el mismo) |

## 9. Lab D3 completo: análisis de ventas con `JOIN`

> **Objetivo del Lab D3:** aplicar todos los conceptos de este manual sobre la base `labd3.db` y responder 5 preguntas de negocio con `JOIN`, `GROUP BY`, `HAVING`, subconsultas y vistas.

### Consulta D3-1: Top 5 clientes por monto total comprado

Técnica: `INNER JOIN` ×2 + `GROUP BY` + `ORDER BY` + `LIMIT`. Tres clientes empatan en S/ 500; el desempate por nombre decide quién entra al Top 5.

In [27]:
pd.read_sql_query("""
/* LAB D3 — Consulta 1: Top 5 clientes por monto total comprado */
SELECT c.nombre                    AS Cliente,
       c.ciudad                    AS Ciudad,
       COUNT(p.id_pedido)          AS Num_Pedidos,      -- cuántas veces compró
       SUM(p.cantidad)             AS Unidades_Total,   -- total de unidades
       SUM(p.cantidad * pr.precio) AS Monto_Total,      -- monto en soles
       MAX(p.fecha)                AS Ultima_Compra     -- fecha de la última compra
FROM   pedidos p
INNER JOIN clientes  c  ON p.id_cliente  = c.id_cliente
INNER JOIN productos pr ON p.id_producto = pr.id_producto
GROUP BY c.id_cliente, c.nombre, c.ciudad
ORDER BY Monto_Total DESC, Cliente                      -- empate: orden alfabético
LIMIT 5
""", conn)

,Cliente,Ciudad,Num_Pedidos,Unidades_Total,Monto_Total,Ultima_Compra
0,María Quispe,Cusco,4,8,5140.0,2024-05-20
1,Carlos Ttito,Cusco,3,3,4020.0,2024-06-01
2,Pedro Huanca,Cusco,2,5,1440.0,2024-05-01
3,José Apaza,Cusco,1,4,720.0,2024-03-10
4,Diana Flores,Puno,1,1,500.0,2024-05-10


### Consulta D3-2: Productos más vendidos por categoría

In [28]:
pd.read_sql_query("""
SELECT pr.categoria                 AS Categoria,
       pr.nombre                    AS Producto,
       SUM(p.cantidad)              AS Unidades_Vendidas,
       SUM(p.cantidad * pr.precio)  AS Ingresos
FROM   pedidos p
INNER JOIN productos pr ON p.id_producto = pr.id_producto
GROUP BY pr.categoria, pr.id_producto        -- agrupa por categoría y producto
ORDER BY pr.categoria, Ingresos DESC
""", conn)

,Categoria,Producto,Unidades_Vendidas,Ingresos
0,Hardware,Laptop Dell i7,2,6400.0
1,Hardware,Disco Externo 2TB,1,320.0
2,Servicio,Capacitación SQLite,3,1500.0
3,Servicio,Soporte Técnico,5,1250.0
4,Software,Licencia Office 365,5,2250.0
5,Software,Antivirus Anual,10,1800.0


### Consulta D3-3: Clientes sin pedidos (`LEFT JOIN` + `IS NULL`)

In [29]:
pd.read_sql_query("""
SELECT c.nombre AS Cliente_Inactivo,
       c.ciudad AS Ciudad,
       c.email  AS Email
FROM   clientes c
LEFT JOIN pedidos p ON c.id_cliente = p.id_cliente
WHERE  p.id_pedido IS NULL           -- NULL = no tiene ningún pedido
""", conn)

,Cliente_Inactivo,Ciudad,Email
0,Marco Salas,Cusco,None


### Consulta D3-4: Análisis mensual de ventas

In [30]:
pd.read_sql_query("""
SELECT SUBSTR(p.fecha, 1, 7)                    AS Mes,
       COUNT(DISTINCT p.id_cliente)             AS Clientes_Activos,   -- clientes distintos
       COUNT(p.id_pedido)                       AS Num_Pedidos,
       SUM(p.cantidad * pr.precio)              AS Ventas_Mes,
       ROUND(AVG(p.cantidad * pr.precio), 2)    AS Ticket_Promedio     -- gasto promedio por pedido
FROM   pedidos p
INNER JOIN productos pr ON p.id_producto = pr.id_producto
GROUP BY SUBSTR(p.fecha, 1, 7)
ORDER BY Mes
""", conn)

,Mes,Clientes_Activos,Num_Pedidos,Ventas_Mes,Ticket_Promedio
0,2024-01,2,3,4640.0,1546.67
1,2024-02,3,3,1650.0,550.00
2,2024-03,3,3,4240.0,1413.33
3,2024-04,2,2,950.0,475.00
4,2024-05,3,3,1540.0,513.33
5,2024-06,1,1,500.0,500.00


### Consulta D3-5: Reporte final con la vista y segmentación

Usa la vista `v_ventas_cliente` creada en la sección 7.

In [31]:
pd.read_sql_query("""
SELECT Cliente,
       Ciudad,
       Total_Pedidos,
       COALESCE(Total_Comprado, 0) AS Total_Comprado,
       Ultima_Compra,
       CASE
           WHEN Total_Comprado >= 5000 THEN 'VIP'
           WHEN Total_Comprado >= 2000 THEN 'Frecuente'
           WHEN Total_Comprado >  0    THEN 'Ocasional'
           ELSE                             'Sin compras'
       END AS Segmento
FROM   v_ventas_cliente
ORDER BY Total_Comprado DESC, Cliente
""", conn)

,Cliente,Ciudad,Total_Pedidos,Total_Comprado,Ultima_Compra,Segmento
0,María Quispe,Cusco,4,5140.0,2024-05-20,VIP
1,Carlos Ttito,Cusco,3,4020.0,2024-06-01,Frecuente
2,Pedro Huanca,Cusco,2,1440.0,2024-05-01,Ocasional
3,José Apaza,Cusco,1,720.0,2024-03-10,Ocasional
4,Diana Flores,Puno,1,500.0,2024-05-10,Ocasional
5,Lucía Condori,Arequipa,1,500.0,2024-02-18,Ocasional
6,Sofía Ccopa,Lima,1,500.0,2024-04-05,Ocasional
7,Roberto Huallpa,Cusco,1,450.0,2024-04-15,Ocasional
8,Ana Mamani,Puno,1,250.0,2024-02-05,Ocasional
9,Marco Salas,Cusco,0,0.0,NaN,Sin compras


## 10. ¿Cuándo usar `INNER JOIN` y cuándo `LEFT JOIN`?

| Pregunta de negocio | Unión recomendada | ¿Por qué? |
|---|---|---|
| ¿Qué clientes han comprado? | `INNER JOIN` | Solo los que tienen pedidos |
| ¿Todos los clientes, hayan comprado o no? | `LEFT JOIN` | Incluye clientes sin pedidos |
| ¿Qué productos se han vendido alguna vez? | `INNER JOIN` | Solo los que aparecen en pedidos |
| ¿Todos los productos, incluso los no vendidos? | `LEFT JOIN` | Muestra productos sin ventas |
| Reporte de ventas del mes | `INNER JOIN` | Solo transacciones reales |
| Buscar registros huérfanos (sin relación) | `LEFT JOIN` + `IS NULL` | Detecta datos sin coincidencia |
| Comparar listas de dos tablas | `FULL JOIN` (o `UNION` de dos `LEFT JOIN`) | Ver las diferencias entre tablas |

> **Pregunta de reflexión.** ¿Por qué en la Consulta D3-1 usamos `INNER JOIN` y no `LEFT JOIN`?
> → Porque solo queremos clientes **que sí compraron**: un cliente sin pedidos no puede tener monto total.
>
> ¿Y si quisiéramos ver también a los clientes que nunca compraron?
> → Usaríamos `LEFT JOIN` y `COALESCE(SUM(...), 0)` para que aparezcan con S/ 0.

## 11. Resumen de comandos avanzados

| Comando | ¿Para qué sirve? | Ejemplo |
|---|---|---|
| `INNER JOIN` | Solo filas con coincidencia en ambas tablas | `JOIN pedidos p ON c.id_cliente = p.id_cliente` |
| `LEFT JOIN` | Todas las de la izquierda + coincidencias | `LEFT JOIN pedidos p ON ...` |
| `RIGHT JOIN` / `FULL JOIN` | Todo de la derecha / de ambas (SQLite 3.39 o mayor) | `FULL JOIN curso k ON ...` |
| `UNION` / `UNION ALL` | Une el resultado de dos `SELECT` (sin / con repetidos) | `SELECT a UNION ALL SELECT b` |
| `GROUP BY` | Agrupa filas para las funciones de resumen | `GROUP BY ciudad` |
| `HAVING` | Filtra grupos (después de `GROUP BY`) | `HAVING SUM(monto) > 1000` |
| Subconsulta en `WHERE`/`HAVING` | Filtra con el resultado de otro `SELECT` | `WHERE id IN (SELECT ...)` |
| Subconsulta en `FROM` | Usa un `SELECT` como tabla temporal | `FROM (SELECT ...) AS t` |
| `CASE WHEN` | Condicional dentro de un `SELECT` | `CASE WHEN x > 5 THEN 'A' ELSE 'B' END` |
| `CREATE VIEW` | Guarda una consulta con un nombre reutilizable | `CREATE VIEW v AS SELECT ...` |
| `COALESCE(x, 0)` | Reemplaza `NULL` por un valor por defecto | `COALESCE(SUM(monto), 0)` |
| `SUBSTR(texto, inicio, largo)` | Extrae parte de un texto | `SUBSTR('2024-05-10', 1, 7)` |
| `ROUND(valor, n)` | Redondea a `n` decimales | `ROUND(AVG(precio), 2)` |

## 12. Ejercicios

Escribe tu consulta en una celda nueva con `pd.read_sql_query("""...""", conn)`. Cada ejercicio indica qué debe devolver. La solución está escondida.

---

**Ejercicio 1.** Muestra el **total comprado por cada cliente de Cusco** que tenga pedidos, de mayor a menor. *Debe devolver 5 filas; el primero es María Quispe con 5140.0.*

<details><summary>Ver solución</summary>

```python
pd.read_sql_query("""
SELECT c.nombre AS Cliente, SUM(p.cantidad * pr.precio) AS Total_Comprado
FROM   pedidos p
INNER JOIN clientes  c  ON p.id_cliente  = c.id_cliente
INNER JOIN productos pr ON p.id_producto = pr.id_producto
WHERE  c.ciudad = 'Cusco'
GROUP BY c.id_cliente
ORDER BY Total_Comprado DESC
""", conn)
```
</details>

---

**Ejercicio 2.** Muestra los productos de los que se vendieron **más de 4 unidades** en total. *Debe devolver 3 filas: Antivirus Anual (10), Licencia Office 365 (5) y Soporte Técnico (5).*

<details><summary>Ver solución</summary>

```python
pd.read_sql_query("""
SELECT pr.nombre AS Producto, SUM(p.cantidad) AS Unidades
FROM   pedidos p
INNER JOIN productos pr ON p.id_producto = pr.id_producto
GROUP BY pr.id_producto
HAVING SUM(p.cantidad) > 4
ORDER BY Unidades DESC, Producto
""", conn)
```
</details>

---

**Ejercicio 3.** Muestra los clientes cuyo total comprado **supera S/ 1 000**. *Debe devolver 3 filas: María Quispe, Carlos Ttito y Pedro Huanca.*

<details><summary>Ver solución</summary>

```python
pd.read_sql_query("""
SELECT c.nombre AS Cliente, SUM(p.cantidad * pr.precio) AS Total_Comprado
FROM   pedidos p
INNER JOIN clientes  c  ON p.id_cliente  = c.id_cliente
INNER JOIN productos pr ON p.id_producto = pr.id_producto
GROUP BY c.id_cliente
HAVING SUM(p.cantidad * pr.precio) > 1000
ORDER BY Total_Comprado DESC
""", conn)
```
</details>

---

**Ejercicio 4.** Con una **subconsulta**, muestra los productos cuyo precio es **mayor que el precio promedio** de todos los productos. *Debe devolver 1 fila: Laptop Dell i7.*

<details><summary>Ver solución</summary>

```python
pd.read_sql_query("""
SELECT nombre, precio
FROM   productos
WHERE  precio > (SELECT AVG(precio) FROM productos)
""", conn)
```
</details>

---

**Ejercicio 5.** ¿Qué **ciudad** concentra más pedidos? *Debe devolver 1 fila: Cusco con 11 pedidos.*

<details><summary>Ver solución</summary>

```python
pd.read_sql_query("""
SELECT c.ciudad AS Ciudad, COUNT(p.id_pedido) AS Pedidos
FROM   pedidos p
INNER JOIN clientes c ON p.id_cliente = c.id_cliente
GROUP BY c.ciudad
ORDER BY Pedidos DESC
LIMIT 1
""", conn)
```
</details>

---

## 13. Limpieza (opcional)

Si quieres dejar la base sin las vistas y las tablas de práctica, ejecuta esta celda. Las vistas se borran **al final** porque la consulta D3-5 las necesita.

In [32]:
conn.executescript("""
DROP VIEW  IF EXISTS v_productos_ranking;
DROP VIEW  IF EXISTS v_ventas_cliente;
DROP TABLE IF EXISTS taller;
DROP TABLE IF EXISTS curso;
""")
print("Vistas y tablas de práctica eliminadas (los datos de clientes, productos y pedidos siguen intactos)")

Vistas y tablas de práctica eliminadas (los datos de clientes, productos y pedidos siguen intactos)


---

*SQLite Avanzado · TECSUP · Fundamentos de Gestión de Datos · Docente: Pilar Rocío Sayán Mejía*